In [0]:
import subprocess
subprocess.check_call(['pip', 'install', 'faker', 'holidays'])

#from utils import save_to_parquet
import numpy as np, pandas as pd, random
from faker import Faker
import holidays

np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# Get catalog and schema from notebook widgets / job parameters
# Widgets are automatically created from base_parameters in the workflow
# For local or ad‑hoc runs, defaults are used
dbutils.widgets.text("CATALOG_NAME", "your_workspace_catalog", "Catalog")
dbutils.widgets.text("SCHEMA_NAME", "glucosphere", "Schema")

try:
    CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
    SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
except Exception as e:
    raise RuntimeError(f"Widget lookup failed, catalog/schema not passed to notebook: {e}")

print(f"Using catalog: {CATALOG_NAME}, schema: {SCHEMA_NAME}")

# Ensure Unity Catalog structure exists for data storage
print("Setting up Unity Catalog structure...")
try:
    # Ensure catalog exists (if you don't have permission to create catalogs,
    # comment this out and create the catalog once via SQL instead)
    #spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    print(f"✓ Catalog '{CATALOG_NAME}' ready")
    
    # Ensure schema exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    print(f"✓ Schema '{SCHEMA_NAME}' ready")
    
    # NOW set current catalog and schema (after creating them)
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    
    # Ensure volume exists
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.pipeline_data
        COMMENT 'Landing zone for MedTech raw data files'
    """)
    print("✓ Volume 'pipeline_data' ready")
    
    # Create subdirectories for all datasets using dbutils
    volume_base = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/pipeline_data"
    datasets = [
        "raw_patient_registry",
        "raw_device_telemetry_stream"
    ]
    
    for dataset in datasets:
        dataset_path = f"{volume_base}/{dataset}"
        try:
            dbutils.fs.mkdirs(dataset_path)
            print(f"✓ Volume directory '{dataset}' created")
        except Exception as dir_error:
            print(f"⚠ Could not create directory {dataset}: {dir_error}")
    
except Exception as e:
    print(f"⚠ Could not create Unity Catalog structure: {e}")
    print("  You may need to create catalog/schema/volume manually before running this script")


In [ ]:
%run ./_device_model_spec

In [ ]:
# Device & patient settings
PATIENT_START = pd.Timestamp("2022-06-01").floor("ms")
PATIENT_END   = pd.Timestamp("2025-11-16").floor("ms")
REGIONS = ["NA", "EMEA", "APAC"]
REGION_PROBS = np.array([0.45, 0.32, 0.23])
# DEVICE_MODELS + their population weights come from _device_model_spec (%run above) —
# single source of truth shared with 05 + the telemetry generator. device_model is now
# assigned deterministically from patient_id, not drawn here.
DIAGNOSES = ["T1D", "T2D", "gestational"]
DIAG_PROBS = np.array([0.45, 0.48, 0.07])
#FIRMWARE_BASE = "v3.14"
#FIRMWARE_EVENT = "v4.00"
#FIRMWARE_FIX = "v4.10"
COVERAGE_TYPES = ["insurance", "public", "self-pay"]
COVERAGE_PROBS = np.array([0.68, 0.22, 0.10])
CARE_EPISODE_TYPES = ["intervention", "routine visit", "telehealth", "acute admission", "icu"]
CARE_EP_TYPE_PROBS = np.array([0.39, 0.34, 0.18, 0.07, 0.02])

In [ ]:
# ========== Patient Registry ==========
# device_model is NOT drawn here — it is a deterministic function of patient_id,
# assigned via the shared _device_model_spec in the write cell below, so it matches
# 05 (incident cohorts) + the telemetry generator exactly. Every other per-patient
# attribute is drawn positionally over a DETERMINISTICALLY SORTED patient list (with
# np.random.seed(42) set above), so the whole registry is reproducible run-to-run.
def generate_patient_registry(n_patients=1000):
    import pyspark.sql.functions as _F
    device_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_device")
    print(f"Generating patient_registry ...")
    # Pull patient_id <-> device_id PAIRS together (one device per patient), sorted by
    # patient_id. Previously patient_id and device_id came from two SEPARATE
    # distinct().toPandas() calls and were zipped by position, so a different row order
    # in each could mis-pair them; sorting + pairing fixes that and makes the positional
    # attribute draws below stable.
    pairs = (device_df.groupBy("patient_id")
             .agg(_F.min("device_id").alias("device_id"))
             .orderBy("patient_id")
             .toPandas())
    patient_id = pairs["patient_id"].tolist()
    device_id  = pairs["device_id"].tolist()
    n_patients = len(patient_id)

    region = np.random.choice(REGIONS, size=n_patients, p=REGION_PROBS)
    sample_birth_years = np.concatenate([
            np.random.randint(1940, 1958, size=int(n_patients*0.18)),   # 65+
            np.random.randint(1958, 1984, size=int(n_patients*0.43)),   # 41-64
            np.random.randint(1984, 2007, size=int(n_patients*0.32)),   # 18-40
            np.random.randint(2007, 2022, size=int(n_patients*0.07)),   # <18
        ])
    if len(sample_birth_years) < n_patients:
        sample_birth_years = np.resize(sample_birth_years, n_patients)
    birth_year = np.random.choice(sample_birth_years, size=n_patients, replace=True)

    # Age-conditioned diagnosis (clinical plausibility): T1D any age; T2D adult-onset
    # (>=18); gestational only childbearing age (18-50). Prevents implausible combos
    # like a 9yo with T2D or post-menopausal gestational diabetes. REF_YEAR matches the
    # birth-year buckets above (age = REF_YEAR - birth_year). Supersedes the old
    # independent DIAG_PROBS draw, so overall marginals shift slightly toward this
    # age-realistic mix.
    REF_YEAR = 2025
    _age = REF_YEAR - birth_year
    diagnosis = np.empty(n_patients, dtype=object)
    _minor = _age < 18
    _childbearing = (_age >= 18) & (_age <= 50)
    _older = _age > 50
    diagnosis[_minor] = "T1D"  # pediatric -> T1D (T2D<18 rare; gestational impossible)
    diagnosis[_childbearing] = np.random.choice(["T1D", "T2D", "gestational"], size=int(_childbearing.sum()), p=[0.40, 0.45, 0.15])
    diagnosis[_older] = np.random.choice(["T1D", "T2D"], size=int(_older.sum()), p=[0.30, 0.70])  # no gestational >50

    # Activation dates are spread
    activation_base = pd.date_range(PATIENT_START - pd.Timedelta(days=90), PATIENT_START, freq="D")
    activation_date = np.random.choice(activation_base, size=n_patients)
    df = pd.DataFrame({
        "patient_id": patient_id,
        "device_id": device_id,
        "region": region,
        "diagnosis": diagnosis,
        "activation_date": pd.to_datetime(activation_date).normalize(),
        "birth_year": birth_year,
        # device_model intentionally omitted — added deterministically in Spark
        # (device_model_case_sql) in the write cell below.
    })
    print("Patient registry generated (device_model assigned deterministically in write cell).")
    return df

In [ ]:
from pyspark.sql.functions import count_distinct, collect_list
from pyspark.sql import functions as F

raw_data_name = "raw_patient_registry"
path = f"{volume_base}/{raw_data_name}"  

device_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_device")
n_patients = device_df.agg(count_distinct("patient_id")).collect()[0][0]

patient_registry = generate_patient_registry(n_patients=n_patients)

# device_model: deterministic function of patient_id via the shared _device_model_spec
# (%run'd above) — IDENTICAL to 05's incident cohorts and the telemetry generator, so
# the gold device_model SSOT, the calibration-bias cohorts, and the firmware timeline
# all agree by construction (no random draw, no run-to-run drift).
patient_registry_sdf = (
    spark.createDataFrame(patient_registry)
         .withColumn("device_model", F.expr(device_model_case_sql("patient_id")))
)
patient_registry_sdf.write.mode("overwrite").parquet(path)
display(patient_registry_sdf)